# Codex Python SDK — Advanced Notebook

This notebook is built for advanced, production-style usage:
- one-time environment bootstrap
- resilient sync flows
- concurrent async flows
- multimodal with graceful fallback
- explicit error handling patterns


In [ ]:
# Single bootstrap cell: ensure local SDK + dependencies are available in this kernel.
import sys, subprocess
from pathlib import Path

repo_python_dir = Path.cwd().resolve().parent  # notebooks/ -> sdk/python
src_dir = repo_python_dir / 'src'

try:
    import pydantic  # noqa: F401
    import codex_app_server  # noqa: F401
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(repo_python_dir)])

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print('Kernel:', sys.executable)
print('SDK src:', src_dir)


In [ ]:
from codex_app_server import Codex, TextInput, ImageInput, LocalImageInput
from codex_app_server.retry import retry_on_overload
from codex_app_server.errors import (
    JsonRpcError,
    InvalidParamsError,
    MethodNotFoundError,
    ServerBusyError,
)


## 1) Resilient sync flow (retry + structured fallback)


In [ ]:
with Codex() as codex:
    thread = codex.thread_start(model='gpt-5')

    result = retry_on_overload(
        lambda: thread.turn(TextInput('Give 5 bullets on robust SDK design.')).run(),
        max_attempts=3,
        initial_delay_s=0.25,
        max_delay_s=2.0,
    )

    print('status=', result.status)
    print(result.text[:500])


## 2) Multimodal with graceful fallback

If remote image fetch/model handling fails, we catch and continue.


In [ ]:
from base64 import b64decode
from pathlib import Path

sample_img = Path.cwd() / 'sample.png'
if not sample_img.exists():
    sample_img.write_bytes(
        b64decode('iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAwMCAO7Z4xQAAAAASUVORK5CYII=')
    )

with Codex() as codex:
    thread = codex.thread_start(model='gpt-5')

    try:
        remote = thread.turn([
            TextInput('Describe this image in 2 bullets.'),
            ImageInput('https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg'),
        ]).run()
        print('remote vision status=', remote.status)
        print((remote.text or '')[:250])
    except Exception as exc:
        print('remote vision failed (expected in some environments):', exc)

    local = thread.turn([
        TextInput('Describe this local image in 2 bullets.'),
        LocalImageInput(str(sample_img)),
    ]).run()
    print('local vision status=', local.status)
    print((local.text or '')[:250])


## 3) Async concurrent turns (fan-out)


In [ ]:
import asyncio
from codex_app_server.async_client import AsyncAppServerClient


async def run_one(prompt: str) -> str:
    async with AsyncAppServerClient() as client:
        await client.initialize()
        started = await client.thread_start(model='gpt-5')
        tid = started['thread']['id']
        turn = await client.turn_text(tid, prompt)
        turn_id = turn['turn']['id']

        chunks = []
        while True:
            evt = await client.next_notification()
            if evt.method == 'item/agentMessage/delta':
                chunks.append((evt.params or {}).get('delta', ''))
            if evt.method == 'turn/completed' and (evt.params or {}).get('turn', {}).get('id') == turn_id:
                break
        return ''.join(chunks).strip()


async def concurrent_demo():
    prompts = [
        'One sentence about retries.',
        'One sentence about backpressure.',
        'One sentence about idempotency.',
    ]
    outs = await asyncio.gather(*(run_one(p) for p in prompts), return_exceptions=True)
    for i, out in enumerate(outs, 1):
        print(f'[{i}]', out)


await concurrent_demo()


## 4) Explicit error handling template


In [ ]:
with Codex() as codex:
    thread = codex.thread_start(model='gpt-5')
    try:
        # normal turn
        ok = thread.turn(TextInput('Return one short line.')).run()
        print('ok status=', ok.status)

        # intentionally trigger method-level error via low-level request
        codex._client.request('demo/missingMethod', {})  # noqa: SLF001

    except MethodNotFoundError as exc:
        print('MethodNotFoundError:', exc.message)
    except InvalidParamsError as exc:
        print('InvalidParamsError:', exc.message)
    except ServerBusyError as exc:
        print('ServerBusyError:', exc.message)
    except JsonRpcError as exc:
        print(f'JsonRpcError {exc.code}:', exc.message)
    except Exception as exc:
        print('Unexpected:', type(exc).__name__, exc)
